# Pipeline conversacional de Chatbot com NLP (chatbotv4)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK, spaCy, Scikit-Learn  
- **Arquivo Original:** `ChatbotV4.ipynb`  

---


## Instalação das Bibliotecas
## Reiniciar o colab após a instalação

In [1]:
!pip install --upgrade pip
!pip install nltk
!pip install spacy
!python -m spacy download pt_core_news_lg
!pip install --upgrade numpy scipy gensim
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 51.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 45.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully 

In [13]:
import nltk
import spacy
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Carregar o modelo do spaCy
nlp = spacy.load("pt_core_news_lg")

## Carregamento do Texto

In [14]:
import requests

url = 'https://raw.githubusercontent.com/VictorSilvaCamargo/o-guarani/main/o-guarani.txt'
response = requests.get(url)

if response.status_code == 200:
    texto_livro = response.text
else:
    raise Exception(f"Erro ao carregar o livro: {response.status_code}")


## Pré Processamento e Geração de Chunks

In [15]:
# --- Pre-processamento ---
def preprocessar_texto(texto):
    texto = texto.lower()
    doc = nlp(texto)
    tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
    return " ".join(tokens)

# --- Geração de Chunks e Extração de Entidades ---
def processar_texto_e_gerar_chunks(texto, tamanho_chunk=200, sobreposicao=50):
    chunks = []
    mapa_chunks = {}
    doc = nlp(texto)
    sentences = [sent.text for sent in doc.sents] # Dividir em sentenças

    current_chunk_tokens = []
    current_chunk_text = []
    current_chunk_entities = {'PER': set(), 'LOC': set(), 'ORG': set(), 'MISC': set()}
    chunk_id_counter = 0

    for i, sent_text in enumerate(sentences):
        sent_doc = nlp(sent_text) # Processar sentença individualmente para entidades e tokens
        sent_tokens = [token.lemma_ for token in sent_doc if token.is_alpha and not token.is_stop]

        # Extrair entidades
        for ent in sent_doc.ents:
            if ent.label_ in current_chunk_entities:
                current_chunk_entities[ent.label_].add(ent.text)

        # Adicionar à lista de tokens e texto do chunk
        current_chunk_tokens.extend(sent_tokens)
        current_chunk_text.append(sent_text)

        # Verificar se o chunk atingiu o tamanho
        if len(current_chunk_tokens) >= tamanho_chunk:
            chunk_text_str = " ".join(current_chunk_text)
            chunk_tokens_str = " ".join(current_chunk_tokens) # Tokens para Word2Vec

            mapa_chunks[chunk_id_counter] = {
                'texto_original': chunk_text_str,
                'tokens_processados': chunk_tokens_str.split(), # Lista de tokens para Word2Vec
                'entidades': {k: list(v) for k, v in current_chunk_entities.items()} # Converte set para list
            }
            chunks.append(chunk_text_str) # Para TF-IDF

            chunk_id_counter += 1

            # Reiniciar para o próximo chunk com sobreposição
            # Mantemos parte do final do chunk anterior como início do próximo
            num_overlap_tokens = len(sent_tokens) # Mantenha a última sentença como sobreposição
            current_chunk_tokens = sent_tokens # Comece o novo chunk com os tokens da última sentença
            current_chunk_text = [sent_text] # Comece o novo chunk com a última sentença
            current_chunk_entities = {'PER': set(), 'LOC': set(), 'ORG': set(), 'MISC': set()}
            for ent in sent_doc.ents: # Re-adiciona entidades da última sentença
                if ent.label_ in current_chunk_entities:
                    current_chunk_entities[ent.label_].add(ent.text)

    # Adicionar o último chunk se houver
    if current_chunk_tokens:
        chunk_text_str = " ".join(current_chunk_text)
        chunk_tokens_str = " ".join(current_chunk_tokens)
        mapa_chunks[chunk_id_counter] = {
            'texto_original': chunk_text_str,
            'tokens_processados': chunk_tokens_str.split(),
            'entidades': {k: list(v) for k, v in current_chunk_entities.items()}
        }
        chunks.append(chunk_text_str)

    return chunks, mapa_chunks

## Treinameto W2V E TF-IDF

In [16]:
# Geração dos chunks
chunks, mapa_chunks = processar_texto_e_gerar_chunks(texto_livro)

# Preparar dados para Word2Vec e TF-IDF
sentencas_tokenizadas_para_w2v = [data['tokens_processados'] for data in mapa_chunks.values()]
textos_originais_para_tfidf = [data['texto_original'] for data in mapa_chunks.values()]

# Treinar Word2Vec
# Ajustar min_count e window se necessário
model_w2v = Word2Vec(sentences=sentencas_tokenizadas_para_w2v, vector_size=100, window=5, min_count=1, workers=4)

# Treinar TF-IDF
# Usar ngram_range para capturar frases (bi-grams)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=list(nlp.Defaults.stop_words)) # Adiciona stop words do spacy

tfidf_matrix = tfidf_vectorizer.fit_transform(textos_originais_para_tfidf)

## Vetor Médio de Respostas

In [17]:
# --- Função para Obter Vetor de Média de Palavras (para Word2Vec) ---
def obter_vetor_media(tokens, model, vector_size):
    vetores = [model.wv[token] for token in tokens if token in model.wv]
    if vetores:
        return np.mean(vetores, axis=0)
    else:
        return np.zeros(vector_size)

## Função de Respostas

In [18]:
# --- Função Principal de Resposta ---
def responder_pergunta(pergunta, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=1):
    pergunta_processada = preprocessar_texto(pergunta).split()
    doc_pergunta = nlp(pergunta.lower()) # Para extração de entidades na pergunta

    # Vetor Word2Vec da pergunta
    vetor_pergunta_w2v = obter_vetor_media(pergunta_processada, model_w2v, model_w2v.vector_size)

    # Vetor TF-IDF da pergunta
    vetor_pergunta_tfidf = tfidf_vectorizer.transform([pergunta])

    scores = []
    for idx, chunk_data in mapa_chunks.items():
        chunk_tokens = chunk_data['tokens_processados']
        chunk_texto_original = chunk_data['texto_original']

        # 1. Similaridade Word2Vec
        vetor_chunk_w2v = obter_vetor_media(chunk_tokens, model_w2v, model_w2v.vector_size)
        if np.all(vetor_pergunta_w2v == 0) or np.all(vetor_chunk_w2v == 0):
            score_w2v = 0.0
        else:
            score_w2v = cosine_similarity(vetor_pergunta_w2v.reshape(1, -1), vetor_chunk_w2v.reshape(1, -1))[0][0]

        # 2. Similaridade TF-IDF
        chunk_tfidf_vector = tfidf_matrix[idx]
        if np.all(vetor_pergunta_tfidf.toarray() == 0) or np.all(chunk_tfidf_vector.toarray() == 0):
            score_tfidf = 0.0
        else:
            score_tfidf = cosine_similarity(vetor_pergunta_tfidf, chunk_tfidf_vector)[0][0]

        # 3. Pontuação por Entidades (Contextualização)
        score_entidades = 0.0
        pergunta_entidades = {'PER': set(), 'LOC': set()} # Focar em Pessoas e Locais da pergunta
        for ent in doc_pergunta.ents:
            if ent.label_ in ['PER', 'LOC']:
                pergunta_entidades[ent.label_].add(ent.text)

        # Aumentar o score se entidades da pergunta estiverem no chunk
        for ent_type, ents_in_query in pergunta_entidades.items():
            for ent_text in ents_in_query:
                # Verifica se a entidade da pergunta está nas entidades do chunk
                if ent_text in ' '.join(chunk_data['entidades'].get(ent_type, [])):
                    score_entidades += 0.2 # Peso arbitrário, pode ser ajustado

        # 4. Combinação de Scores (Ajuste por PER)
        # Ajuste os pesos conforme a importância de cada componente
        # Exemplo: 60% Word2Vec, 30% TF-IDF, 10% Entidades
        score_combinado = (score_w2v * 0.6) + (score_tfidf * 0.3) + score_entidades

        # Ajuste Final: Se uma entidade da pergunta não for encontrada no chunk, penalizar o score
        # Isso é uma forma de garantir que a contextualização funcione melhor.
        entidades_presentes_no_chunk = False
        if pergunta_entidades['PER'] or pergunta_entidades['LOC']:
            for ent_type, ents_in_query in pergunta_entidades.items():
                for ent_text in ents_in_query:
                    if ent_text in ' '.join(chunk_data['entidades'].get(ent_type, [])):
                        entidades_presentes_no_chunk = True
                        break
                if entidades_presentes_no_chunk:
                    break

        if (pergunta_entidades['PER'] or pergunta_entidades['LOC']) and not entidades_presentes_no_chunk:
            score_combinado *= 0.5 # Penaliza se entidades chave não estiverem no chunk

        scores.append((idx, score_combinado))

    # Ordenar por score e selecionar os top N
    selecionados = sorted(scores, key=lambda x: x[1], reverse=True)[:topn]

    if not selecionados or selecionados[0][1] == 0:
        print("\nDesculpe, não consegui encontrar uma resposta relevante no livro.")
        return

    print(f"\nTop {len(selecionados)} respostas (com ajuste por PER):")
    for idx, score in selecionados:
        chunk_data = mapa_chunks[idx]
        print(f"\n🔹 Score (ajustado): {score:.4f}")
        print(chunk_data['texto_original'])
        print(f"   → Pessoas: {', '.join(chunk_data['entidades'].get('PER', [])) or '—'}")
        print(f"   → Locais:   {', '.join(chunk_data['entidades'].get('LOC', [])) or '—'}")

## Interface

In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def setup_interactive_qa(responder_pergunta, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=3):


    # Create the text input widget
    question_input = widgets.Textarea(
        value='',
        placeholder='Digite sua pergunta sobre o livro O Guarani aqui...',
        description='Pergunta:',
        disabled=False,
        layout=widgets.Layout(width='auto', height='80px')
    )

    # Create the output widget to display answers
    output_area = widgets.Output()

    # Create the button to submit the question
    submit_button = widgets.Button(
        description='Responder',
        disabled=False,
        button_style='success', # 'success', 'info', 'warning', 'danger' or ''
        tooltip='Clique para obter a resposta',
        icon='question' # (FontAwesome icons)
    )

    def on_button_click(b):
        with output_area:
            clear_output(wait=True) # Clear previous output
            pergunta_usuario = question_input.value
            if pergunta_usuario:
                print("Buscando resposta para:", pergunta_usuario)
                # Call your existing answer function
                responder_pergunta(pergunta_usuario, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=topn)
            else:
                print("Por favor, digite uma pergunta.")
            question_input.value = '' # Clear the input field after submission

    submit_button.on_click(on_button_click)

    # Arrange the widgets
    ui = widgets.VBox([question_input, submit_button, output_area])
    display(ui)



## Execução

In [20]:
setup_interactive_qa(responder_pergunta, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=3)